# Mask2Former for TWSI Segmentation
---

This notebook trains a Mask2Former model for tactile walking surface indicator
segmentation using the Detectron2 framework.

Mask2Former is a transformer-based universal segmentation model that achieves
state-of-the-art performance on instance, semantic, and panoptic segmentation.

**Training:** ~355K iterations with AdamW optimizer, 1024×1024 crops, AMP enabled.

**Paper Reference:** GuideTWSI (ICRA 2026), Section VI-A

## 1. Installation

Install Detectron2 and Mask2Former dependencies.

In [ ]:
# Install Detectron2 (check https://detectron2.readthedocs.io/en/latest/tutorials/install.html)
!pip install 'git+https://github.com/facebookresearch/detectron2.git' -q

# Clone Mask2Former
!git clone https://github.com/facebookresearch/Mask2Former.git
%cd Mask2Former
!pip install -r requirements.txt -q
!cd mask2former/modeling/pixel_decoder/ops && sh make.sh

## 2. Configuration

Load training hyperparameters from the config file.

In [ ]:
import yaml

with open("../configs/mask2former.yaml", "r") as f:
    config = yaml.safe_load(f)

MAX_ITER = config["training"]["max_iterations"]  # 355000
BATCH_SIZE = config["training"]["batch_size"]  # 2
IMAGE_SIZE = config["training"]["image_size"]  # 1024
BASE_LR = config["training"]["base_learning_rate"]  # 0.0001
AMP = config["training"]["amp_enabled"]  # True

print(f"Training: max_iter={MAX_ITER}, batch={BATCH_SIZE}, lr={BASE_LR}, amp={AMP}")

## 3. Download Dataset

Download the GuideTWSI dataset from HuggingFace Hub.

Mask2Former requires COCO-format annotations. Use `data_utils/format_converters.py`
to convert if needed.

In [ ]:
# Download dataset from HuggingFace
# !huggingface-cli download guidedogrobot-tactile/GuideTWSI --local-dir ./data

# Dataset paths (COCO format)
DATASET_ROOT = "data"
TRAIN_JSON = "data/annotations/train.json"
VAL_JSON = "data/annotations/val.json"
TEST_JSON = "data/annotations/test.json"
TRAIN_IMAGES = "data/train"
VAL_IMAGES = "data/val"
TEST_IMAGES = "data/test"

## 4. Register Dataset with Detectron2

In [ ]:
import os
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

# Register datasets
register_coco_instances("twsi_train", {}, TRAIN_JSON, TRAIN_IMAGES)
register_coco_instances("twsi_val", {}, VAL_JSON, VAL_IMAGES)
register_coco_instances("twsi_test", {}, TEST_JSON, TEST_IMAGES)

# Set metadata
for split in ["twsi_train", "twsi_val", "twsi_test"]:
    MetadataCatalog.get(split).set(thing_classes=["tactile_paving"])

print(f"Train: {len(DatasetCatalog.get('twsi_train'))} images")
print(f"Val: {len(DatasetCatalog.get('twsi_val'))} images")

## 5. Training

Train Mask2Former using the Detectron2 trainer with standard configuration.

**Hyperparameters from paper:**
- ~355K iterations with AdamW optimizer
- 1024×1024 large-scale jittering augmentation crops
- Automatic mixed precision (AMP) enabled

In [ ]:
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.projects.deeplab import add_deeplab_config

# Import Mask2Former config
from mask2former import add_maskformer2_config


def setup_cfg():
    cfg = get_cfg()
    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)

    # Load base Mask2Former config
    cfg.merge_from_file(
        "configs/coco/instance-segmentation/swin/"
        "maskformer2_swin_base_IN21k_384_bs16_50ep.yaml"
    )

    # Dataset
    cfg.DATASETS.TRAIN = ("twsi_train",)
    cfg.DATASETS.TEST = ("twsi_val",)

    # Training
    cfg.SOLVER.MAX_ITER = MAX_ITER
    cfg.SOLVER.BASE_LR = BASE_LR
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.AMP.ENABLED = AMP

    # Model
    cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 1
    cfg.MODEL.MASK_FORMER.NUM_OBJECT_QUERIES = 100

    # Input
    cfg.INPUT.MIN_SIZE_TRAIN = (IMAGE_SIZE,)
    cfg.INPUT.MAX_SIZE_TRAIN = IMAGE_SIZE
    cfg.INPUT.CROP.ENABLED = True
    cfg.INPUT.CROP.SIZE = [1.0, 1.0]

    # Output
    cfg.OUTPUT_DIR = "output/mask2former_twsi"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    return cfg


cfg = setup_cfg()
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## 6. Inference

Run inference on test images and visualize predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer

# Load trained model
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

# Visualize predictions
test_images_list = sorted(os.listdir(TEST_IMAGES))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
metadata = MetadataCatalog.get("twsi_test")

for ax, img_name in zip(axes.flat, test_images_list):
    if not img_name.endswith((".jpg", ".png")):
        continue
    img_path = os.path.join(TEST_IMAGES, img_name)
    img = np.array(Image.open(img_path).convert("RGB"))

    outputs = predictor(img[:, :, ::-1])  # RGB -> BGR for Detectron2
    v = Visualizer(img, metadata, scale=1.0)
    vis = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    ax.imshow(vis.get_image())
    ax.set_title(img_name, fontsize=8)
    ax.axis("off")

plt.suptitle("Mask2Former Predictions", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Evaluation

Evaluate using both Detectron2 built-in evaluator and custom binary metrics.

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator("twsi_test", output_dir=cfg.OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, "twsi_test")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

print("\nDetectron2 COCO Evaluation:")
print(results)

In [ ]:
import sys
from tqdm import tqdm

sys.path.insert(0, "..")
from evaluation.metrics import compute_binary_metrics, aggregate_metrics

test_mask_dir = "data/test/masks"
all_metrics = []

for img_name in tqdm(sorted(os.listdir(TEST_IMAGES)), desc="Evaluating"):
    if not img_name.endswith((".jpg", ".png")):
        continue

    img_path = os.path.join(TEST_IMAGES, img_name)
    mask_path = os.path.join(test_mask_dir, img_name.replace(".jpg", ".png"))
    if not os.path.exists(mask_path):
        continue

    img = np.array(Image.open(img_path).convert("RGB"))
    gt_mask = np.array(Image.open(mask_path).convert("L"))

    outputs = predictor(img[:, :, ::-1])
    instances = outputs["instances"].to("cpu")

    pred_mask = np.zeros_like(gt_mask, dtype=np.uint8)
    if len(instances) > 0 and instances.has("pred_masks"):
        for mask in instances.pred_masks:
            m = mask.numpy().astype(np.uint8) * 255
            if m.shape != pred_mask.shape:
                m = np.array(Image.fromarray(m).resize((gt_mask.shape[1], gt_mask.shape[0])))
            pred_mask = np.maximum(pred_mask, m)

    metrics = compute_binary_metrics(gt_mask, pred_mask)
    all_metrics.append(metrics)

avg = aggregate_metrics(all_metrics)
print(f"\nBinary Segmentation Results ({len(all_metrics)} images):")
for k, v in avg.items():
    print(f"  {k}: {v:.4f}")